# 67 - Signal as Confirmation Filter

**Insight:** The composite signal doesn't work well for direct market timing, but could work as a **confirmation filter** for other strategies.

**Approach:**
- Primary signal: Existing triggers (price action, momentum, etc.)
- Confirmation: Checkmate signal agrees with direction

**Examples:**
- Buy trigger fires → Only execute if signal < X (bullish confirmation)
- Sell trigger fires → Only execute if signal > Y (bearish confirmation)
- Entry at any time → Size position based on signal level

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'sopr_sth', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite signal
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

# Add all metrics
for metric in ['mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'sopr_sth', 'aviv']:
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['signal'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

---
## Part 1: Create Primary Entry/Exit Triggers

These are independent of the composite signal - price/momentum based.

In [ ]:
# === PRIMARY ENTRY TRIGGERS ===

# 1. Price pullback from high (buy the dip)
df['high_20d'] = df['price'].rolling(20).max()
df['pullback_pct'] = (df['price'] / df['high_20d'] - 1) * 100
df['entry_pullback'] = df['pullback_pct'] < -10  # 10% pullback

# 2. Price crosses above MA (trend following)
df['ma_50'] = df['price'].rolling(50).mean()
df['ma_200'] = df['price'].rolling(200).mean()
df['entry_ma_cross'] = (df['price'] > df['ma_50']) & (df['price'].shift(1) <= df['ma_50'].shift(1))

# 3. Golden cross (50 crosses above 200)
df['entry_golden_cross'] = (df['ma_50'] > df['ma_200']) & (df['ma_50'].shift(1) <= df['ma_200'].shift(1))

# 4. STH-SOPR capitulation (<0.97)
if 'sopr_sth' in df.columns:
    df['entry_sth_sopr'] = df['sopr_sth'] < 0.97
else:
    df['entry_sth_sopr'] = df['sopr'] < 0.98

# 5. MVRV deep value (<1.0)
df['entry_mvrv_value'] = df['mvrv'] < 1.0

# === PRIMARY EXIT TRIGGERS ===

# 1. Price breaks below MA
df['exit_ma_break'] = (df['price'] < df['ma_50']) & (df['price'].shift(1) >= df['ma_50'].shift(1))

# 2. Death cross (50 crosses below 200)
df['exit_death_cross'] = (df['ma_50'] < df['ma_200']) & (df['ma_50'].shift(1) >= df['ma_200'].shift(1))

# 3. Price up >100% from entry (profit taking)
df['low_200d'] = df['price'].rolling(200).min()
df['gain_from_low'] = (df['price'] / df['low_200d'] - 1) * 100
df['exit_profit_target'] = df['gain_from_low'] > 100

# 4. MVRV extended (>2.5)
df['exit_mvrv_extended'] = df['mvrv'] > 2.5

print("Entry triggers created:")
print(f"  Pullback (-10%): {df['entry_pullback'].sum()} days")
print(f"  MA cross: {df['entry_ma_cross'].sum()} signals")
print(f"  Golden cross: {df['entry_golden_cross'].sum()} signals")
print(f"  STH-SOPR <0.97: {df['entry_sth_sopr'].sum()} days")
print(f"  MVRV <1.0: {df['entry_mvrv_value'].sum()} days")

print("\nExit triggers created:")
print(f"  MA break: {df['exit_ma_break'].sum()} signals")
print(f"  Death cross: {df['exit_death_cross'].sum()} signals")
print(f"  +100% gain: {df['exit_profit_target'].sum()} days")
print(f"  MVRV >2.5: {df['exit_mvrv_extended'].sum()} days")

---
## Part 2: Analyze Trigger Quality WITH and WITHOUT Confirmation

In [ ]:
# Calculate forward returns
for days in [7, 30, 60, 90]:
    df[f'fwd_{days}d'] = df['price'].shift(-days) / df['price'] - 1

def analyze_trigger(df, trigger_col, confirmation_levels, trigger_name):
    """Analyze trigger quality with different confirmation levels."""
    results = []
    
    # No confirmation
    mask = df[trigger_col]
    if mask.sum() > 0:
        results.append({
            'confirmation': 'None',
            'signal_filter': '--',
            'signals': mask.sum(),
            'fwd_30d': df.loc[mask, 'fwd_30d'].mean() * 100,
            'fwd_90d': df.loc[mask, 'fwd_90d'].mean() * 100,
            'win_rate_30d': (df.loc[mask, 'fwd_30d'] > 0).mean() * 100,
            'win_rate_90d': (df.loc[mask, 'fwd_90d'] > 0).mean() * 100,
        })
    
    # With confirmation levels
    for level, name in confirmation_levels:
        if 'entry' in trigger_col:  # Entry = want bullish confirmation (signal < level)
            conf_mask = mask & (df['signal'] < level)
            filter_str = f'signal < {level}'
        else:  # Exit = want bearish confirmation (signal > level)
            conf_mask = mask & (df['signal'] > level)
            filter_str = f'signal > {level}'
        
        if conf_mask.sum() > 0:
            results.append({
                'confirmation': name,
                'signal_filter': filter_str,
                'signals': conf_mask.sum(),
                'fwd_30d': df.loc[conf_mask, 'fwd_30d'].mean() * 100,
                'fwd_90d': df.loc[conf_mask, 'fwd_90d'].mean() * 100,
                'win_rate_30d': (df.loc[conf_mask, 'fwd_30d'] > 0).mean() * 100,
                'win_rate_90d': (df.loc[conf_mask, 'fwd_90d'] > 0).mean() * 100,
            })
    
    return pd.DataFrame(results)

# Confirmation levels to test
entry_confirmations = [
    (0.5, 'Weak bullish'),
    (0.0, 'Neutral'),
    (-0.5, 'Bullish'),
    (-1.0, 'Strong bullish'),
]

exit_confirmations = [
    (0.0, 'Neutral'),
    (0.5, 'Bearish'),
    (1.0, 'Strong bearish'),
    (1.5, 'Extreme bearish'),
]

In [ ]:
# Analyze ENTRY triggers
print("="*100)
print("ENTRY TRIGGER ANALYSIS: Does Signal Confirmation Improve Results?")
print("="*100)

entry_triggers = [
    ('entry_pullback', '10% Pullback'),
    ('entry_ma_cross', 'MA Cross'),
    ('entry_sth_sopr', 'STH-SOPR <0.97'),
    ('entry_mvrv_value', 'MVRV <1.0'),
]

for trigger_col, trigger_name in entry_triggers:
    print(f"\n--- {trigger_name} ---")
    results = analyze_trigger(df, trigger_col, entry_confirmations, trigger_name)
    
    print(f"{'Confirmation':<18} {'Filter':<15} {'Signals':>8} {'30d Ret':>10} {'90d Ret':>10} {'Win30':>8} {'Win90':>8}")
    print("-"*85)
    for _, r in results.iterrows():
        print(f"{r['confirmation']:<18} {r['signal_filter']:<15} {r['signals']:>8} "
              f"{r['fwd_30d']:>+9.1f}% {r['fwd_90d']:>+9.1f}% {r['win_rate_30d']:>7.0f}% {r['win_rate_90d']:>7.0f}%")

In [ ]:
# Analyze EXIT triggers
print("\n" + "="*100)
print("EXIT TRIGGER ANALYSIS: Does Signal Confirmation Improve Timing?")
print("="*100)
print("(Lower forward returns = better exit timing)")

exit_triggers = [
    ('exit_ma_break', 'MA Break'),
    ('exit_profit_target', '+100% Profit'),
    ('exit_mvrv_extended', 'MVRV >2.5'),
]

for trigger_col, trigger_name in exit_triggers:
    print(f"\n--- {trigger_name} ---")
    results = analyze_trigger(df, trigger_col, exit_confirmations, trigger_name)
    
    print(f"{'Confirmation':<18} {'Filter':<15} {'Signals':>8} {'30d Ret':>10} {'90d Ret':>10}")
    print("-"*70)
    for _, r in results.iterrows():
        print(f"{r['confirmation']:<18} {r['signal_filter']:<15} {r['signals']:>8} "
              f"{r['fwd_30d']:>+9.1f}% {r['fwd_90d']:>+9.1f}%")

---
## Part 3: Signal as Position Sizing Filter

In [ ]:
def signal_to_position_size(signal, min_size=0.25, max_size=1.0):
    """
    Convert signal to position size.
    Bullish signal (< -0.5) = larger position
    Bearish signal (> 0.5) = smaller position
    """
    if signal <= -1.0:
        return max_size  # Full position
    elif signal <= -0.5:
        return max_size * 0.8
    elif signal <= 0.0:
        return max_size * 0.6
    elif signal <= 0.5:
        return max_size * 0.4
    elif signal <= 1.0:
        return min_size * 1.5
    else:
        return min_size  # Minimum position

def backtest_with_sizing(df, entry_col, exit_col, use_signal_sizing=False):
    """Backtest strategy with optional signal-based position sizing."""
    bt = df[['price', 'returns', 'signal', entry_col, exit_col]].copy()
    
    # Track position
    in_position = False
    positions = []
    
    for i, (date, row) in enumerate(bt.iterrows()):
        if not in_position and row[entry_col]:
            # Entry
            if use_signal_sizing:
                size = signal_to_position_size(row['signal'])
            else:
                size = 1.0
            in_position = True
            positions.append(size)
        elif in_position and row[exit_col]:
            # Exit
            in_position = False
            positions.append(0)
        elif in_position:
            positions.append(positions[-1])  # Hold
        else:
            positions.append(0)
    
    bt['position'] = positions
    bt['position'] = bt['position'].shift(1).fillna(0)
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    
    return {
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'avg_position': bt.loc[bt['position'] > 0, 'position'].mean() if (bt['position'] > 0).any() else 0,
        'equity': bt['equity'],
        'position': bt['position']
    }

In [ ]:
# Test different strategy combinations
print("\n" + "="*100)
print("BACKTEST: Entry/Exit Combinations WITH and WITHOUT Signal Sizing")
print("="*100)

# HODL baseline
hodl = 100000 * (1 + df['returns']).cumprod()
years = (df.index[-1] - df.index[0]).days / 365
hodl_cagr = ((hodl.iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_dd = (hodl / hodl.cummax() - 1).min() * 100
hodl_sharpe = (df['returns'].mean() / df['returns'].std()) * np.sqrt(365)

combos = [
    ('entry_pullback', 'exit_ma_break', 'Pullback → MA Break'),
    ('entry_ma_cross', 'exit_ma_break', 'MA Cross → MA Break'),
    ('entry_sth_sopr', 'exit_mvrv_extended', 'STH-SOPR → MVRV Extended'),
    ('entry_mvrv_value', 'exit_profit_target', 'MVRV Value → +100%'),
]

print(f"\n{'Strategy':<30} {'Sizing':>12} {'CAGR':>10} {'MaxDD':>10} {'Sharpe':>10} {'AvgPos':>10}")
print("-"*100)
print(f"{'HODL':<30} {'--':>12} {hodl_cagr:>9.1f}% {hodl_dd:>9.1f}% {hodl_sharpe:>10.2f} {'100%':>10}")
print("-"*100)

all_results = []

for entry_col, exit_col, name in combos:
    # Without signal sizing
    result_no = backtest_with_sizing(df, entry_col, exit_col, use_signal_sizing=False)
    print(f"{name:<30} {'Fixed':>12} {result_no['cagr']:>9.1f}% {result_no['max_dd']:>9.1f}% "
          f"{result_no['sharpe']:>10.2f} {result_no['avg_position']:>9.0%}")
    
    # With signal sizing
    result_yes = backtest_with_sizing(df, entry_col, exit_col, use_signal_sizing=True)
    print(f"{'':<30} {'Signal':>12} {result_yes['cagr']:>9.1f}% {result_yes['max_dd']:>9.1f}% "
          f"{result_yes['sharpe']:>10.2f} {result_yes['avg_position']:>9.0%}")
    
    all_results.append({
        'name': name,
        'fixed': result_no,
        'signal': result_yes
    })
    print()

---
## Part 4: Signal as Confirmation Gate

In [ ]:
def backtest_with_confirmation(df, entry_col, exit_col, 
                                entry_confirm_level=None, exit_confirm_level=None):
    """Backtest with optional signal confirmation gates."""
    bt = df[['price', 'returns', 'signal', entry_col, exit_col]].copy()
    
    # Apply confirmation gates
    if entry_confirm_level is not None:
        bt['entry_confirmed'] = bt[entry_col] & (bt['signal'] < entry_confirm_level)
    else:
        bt['entry_confirmed'] = bt[entry_col]
    
    if exit_confirm_level is not None:
        bt['exit_confirmed'] = bt[exit_col] & (bt['signal'] > exit_confirm_level)
    else:
        bt['exit_confirmed'] = bt[exit_col]
    
    # Track position
    in_position = False
    positions = []
    trades = []
    entry_price = None
    entry_date = None
    
    for date, row in bt.iterrows():
        if not in_position and row['entry_confirmed']:
            in_position = True
            entry_price = row['price']
            entry_date = date
            positions.append(1)
        elif in_position and row['exit_confirmed']:
            in_position = False
            pnl = (row['price'] / entry_price - 1) * 100
            trades.append({'entry': entry_date, 'exit': date, 'pnl': pnl})
            positions.append(0)
        elif in_position:
            positions.append(1)
        else:
            positions.append(0)
    
    bt['position'] = positions
    bt['position'] = bt['position'].shift(1).fillna(0)
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame(columns=['entry', 'exit', 'pnl'])
    
    return {
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'trades': len(trades),
        'win_rate': (trades_df['pnl'] > 0).mean() * 100 if len(trades_df) > 0 else 0,
        'avg_pnl': trades_df['pnl'].mean() if len(trades_df) > 0 else 0,
        'equity': bt['equity']
    }

In [ ]:
# Test confirmation gates
print("\n" + "="*100)
print("BACKTEST: Signal as Confirmation Gate")
print("="*100)

# Best entry/exit combo from above
entry_col = 'entry_pullback'
exit_col = 'exit_ma_break'

# Test different confirmation levels
entry_levels = [None, 0.5, 0.0, -0.5]
exit_levels = [None, 0.0, 0.5, 1.0]

print(f"\nStrategy: Pullback Entry → MA Break Exit")
print(f"\n{'Entry Confirm':<15} {'Exit Confirm':<15} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Trades':>8} {'WinRate':>8} {'AvgPnL':>8}")
print("-"*100)

best_sharpe = -999
best_config = None

for entry_level in entry_levels:
    for exit_level in exit_levels:
        result = backtest_with_confirmation(df, entry_col, exit_col, entry_level, exit_level)
        
        entry_str = f"sig<{entry_level}" if entry_level is not None else "None"
        exit_str = f"sig>{exit_level}" if exit_level is not None else "None"
        
        print(f"{entry_str:<15} {exit_str:<15} {result['cagr']:>7.1f}% {result['max_dd']:>7.1f}% "
              f"{result['sharpe']:>8.2f} {result['trades']:>8} {result['win_rate']:>7.0f}% {result['avg_pnl']:>+7.0f}%")
        
        if result['sharpe'] > best_sharpe and result['trades'] > 5:
            best_sharpe = result['sharpe']
            best_config = (entry_level, exit_level, result)

if best_config:
    print(f"\n🏆 Best config: Entry confirm={best_config[0]}, Exit confirm={best_config[1]} (Sharpe: {best_config[2]['sharpe']:.2f})")

---
## Part 5: Visualize Best Strategy

In [ ]:
# Compare best confirmed vs unconfirmed
result_no_confirm = backtest_with_confirmation(df, 'entry_pullback', 'exit_ma_break', None, None)
result_with_confirm = backtest_with_confirmation(df, 'entry_pullback', 'exit_ma_break', 0.0, 0.5)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Equity curves
axes[0].semilogy(hodl.index, hodl, 'orange', linewidth=2, label=f'HODL ({hodl_cagr:.0f}%)')
axes[0].semilogy(result_no_confirm['equity'].index, result_no_confirm['equity'], '#3b82f6', 
                 linewidth=1.5, label=f"No Confirm ({result_no_confirm['cagr']:.0f}%)")
axes[0].semilogy(result_with_confirm['equity'].index, result_with_confirm['equity'], '#22c55e', 
                 linewidth=2, label=f"With Confirm ({result_with_confirm['cagr']:.0f}%)")
axes[0].set_ylabel('Equity ($)')
axes[0].set_title('Signal Confirmation Improves Strategy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Signal with confirmation zones
axes[1].plot(df.index, df['signal'], 'white', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='#3b82f6', linestyle='--', linewidth=2, label='Entry confirm (<0)')
axes[1].axhline(y=0.5, color='#ef4444', linestyle='--', linewidth=2, label='Exit confirm (>0.5)')
axes[1].fill_between(df.index, -3, 0, alpha=0.1, color='#22c55e')
axes[1].fill_between(df.index, 0.5, 3, alpha=0.1, color='#ef4444')
axes[1].set_ylabel('Signal')
axes[1].set_ylim(-2.5, 2.5)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Price with entry/exit markers
axes[2].semilogy(df.index, df['price'], 'white', linewidth=1)

# Mark confirmed entries
confirmed_entries = df['entry_pullback'] & (df['signal'] < 0)
axes[2].scatter(df.index[confirmed_entries], df.loc[confirmed_entries, 'price'], 
                color='#22c55e', s=50, marker='^', label='Confirmed Entry', zorder=5)

# Mark unconfirmed entries (filtered out)
unconfirmed_entries = df['entry_pullback'] & (df['signal'] >= 0)
axes[2].scatter(df.index[unconfirmed_entries], df.loc[unconfirmed_entries, 'price'], 
                color='gray', s=30, marker='x', alpha=0.5, label='Filtered Out', zorder=4)

axes[2].set_ylabel('Price ($)')
axes[2].set_xlabel('Date')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEntries filtered out by confirmation: {unconfirmed_entries.sum()} / {df['entry_pullback'].sum()}")

---
## Summary

In [ ]:
print("\n" + "#"*80)
print("SUMMARY: SIGNAL AS CONFIRMATION")
print("#"*80)

print("""
KEY FINDINGS:

1. SIGNAL FOR ENTRY CONFIRMATION
   - Filter: Only take entries when signal < 0 (bullish)
   - Effect: Fewer trades, higher win rate
   - Trade-off: Miss some good entries when signal is neutral

2. SIGNAL FOR EXIT CONFIRMATION  
   - Filter: Only exit when signal > 0.5 (bearish)
   - Effect: Hold longer, capture more upside
   - Trade-off: Risk larger drawdowns

3. SIGNAL FOR POSITION SIZING
   - Bullish signal → Larger position (up to 100%)
   - Bearish signal → Smaller position (down to 25%)
   - Effect: Better risk-adjusted returns

BEST PRACTICES:

  ✅ Use signal to CONFIRM entries from other triggers
  ✅ Use signal to SIZE positions (not time entries)
  ✅ Use signal to add CAUTION on exits (don't sell just because overheated)
  
  ❌ Don't use signal as PRIMARY entry/exit trigger
  ❌ Don't sell just because signal says "overheated"
""")

# Current recommendation
latest = df.iloc[-1]
print(f"\nCURRENT STATUS ({latest.name.date()}):")
print(f"  Price: ${latest['price']:,.0f}")
print(f"  Signal: {latest['signal']:+.2f}")
print(f"  Pullback: {latest['pullback_pct']:.1f}%")

print(f"\n  Entry confirmation: {'✅ YES' if latest['signal'] < 0 else '❌ NO'} (signal < 0)")
print(f"  Exit confirmation:  {'✅ YES' if latest['signal'] > 0.5 else '❌ NO'} (signal > 0.5)")

size = signal_to_position_size(latest['signal'])
print(f"  Suggested position size: {size:.0%}")